# Detection of Random-Interval C2 Beaconing

## Overview

This notebook uses a hybrid approach combining statistical clustering (K-Means) with deep learning (Autoencoder) to detect Command & Control (C2) beaconing patterns in network traffic. The solution analyzes Sysmon logs to identify regular-interval beacons by examining timing statistics and process context.

**Objective:** Detect C2 beaconing by analyzing the regularity of network connections combined with behavioral context of initiating processes.

**Data Requirements:**
- Sysmon Operational logs
- Essential Event IDs: 1 (Process Create), 3 (Network Connection), 22 (DNS Query)
- ProcessGuid field for sessionization
- Data readable from MinIO/S3

**Expected Outputs:**
- Trained K-Means clustering model for shallow anomaly detection
- Trained PyTorch Autoencoder for deep anomaly detection
- Anomaly scores from both models
- Combined ensemble beacon score
- Top beacons ranked by final score
- Confusion matrix and classification metrics

**Model Approach:**
**K-Means Clustering (Shallow):**
- Extract timing features from network sessions
- Train clustering model to identify normal behavior patterns
- Calculate distance to cluster center as anomaly score
- Coefficient of Variation (CV) is key feature: beacons have low CV (regular timing)

**Autoencoder (Deep):**
- Enrich with process context (command line, parent image)
- Train neural network to reconstruct normal traffic
- High reconstruction error = anomalous (candidate beacon)

**Ensemble Fusion:**
- Normalize both scores to 0-1 scale
- Combine with context-weighted formula: `beacon_score = shallow * (1 + deep)`
- This emphasizes timing anomalies while considering behavioral context

**Prerequisites:**
- MinIO/S3 access configured in environment variables
- Spark cluster with GPU access
- Sysmon logs available
- PyTorch with GPU support

**Notes:**
- Developed in test environment; tuning is needed for your dataset
- Real-world data is noisy (e.g., OneDrive, Chrome)
- Aggressive tuning of score fusion logic and model thresholds required
- Can be adapted for other data sources (EDR solutions)
- Threshold of 0.8 used in metrics (adjustable in config)

In [ ]:
# Cell 1: Imports & Environment Setup
# =========================================

import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import udf, col, length, collect_set
from pyspark.sql.types import FloatType, DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, IDF, VectorAssembler, StandardScaler, StringIndexer, MinMaxScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.torch.distributor import TorchDistributor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import s3fs
import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

load_dotenv()

print("✅ Environment loaded")

In [ ]:
# Cell 2: Spark Configuration
# =========================================

SPARK_APP_NAME = "C2_Beacon_Detection"

SPARK_MASTER = os.getenv('SPARK_MASTER')
SPARK_DRIVER_HOST = os.getenv('SPARK_DRIVER_HOST')
SPARK_DRIVER_PORT = int(os.getenv('SPARK_DRIVER_PORT', '7771'))
SPARK_BLOCK_MANAGER_PORT = int(os.getenv('SPARK_BLOCK_MANAGER_PORT', '7772'))
SPARK_UI_PORT = int(os.getenv('SPARK_UI_PORT', '8088'))
SPARK_EXECUTOR_CORES = int(os.getenv('SPARK_EXECUTOR_CORES', '16'))
SPARK_EXECUTOR_MEMORY = os.getenv('SPARK_EXECUTOR_MEMORY', '32G')
SPARK_DRIVER_MEMORY = os.getenv('SPARK_DRIVER_MEMORY', '2g')
SPARK_EXECUTOR_GPU_AMOUNT = os.getenv('SPARK_EXECUTOR_GPU_AMOUNT', '1')
SPARK_TASK_GPU_AMOUNT = os.getenv('SPARK_TASK_GPU_AMOUNT', '1')
SPARK_JARS_PATH = os.getenv('SPARK_JARS_PATH', '/home/jmi/Spark_cluster/master/jars/*')
SPARK_EXECUTOR_CLASSPATH = os.getenv('SPARK_EXECUTOR_CLASSPATH', '/opt/spark/jars/*:/opt/spark/external-jars/*')
USE_GPU = os.getenv('USE_GPU', 'true').lower() == 'true'

print(f"✅ Spark Config: master={SPARK_MASTER}, cores={SPARK_EXECUTOR_CORES}, memory={SPARK_EXECUTOR_MEMORY}, gpu={USE_GPU}")

In [ ]:
# Cell 3: MinIO Configuration
# =========================================

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
MINIO_PATH_STYLE_ACCESS = os.getenv('MINIO_PATH_STYLE_ACCESS', 'true')

STORAGE_OPTIONS = {
    'key': MINIO_ACCESS_KEY,
    'secret': MINIO_SECRET_KEY,
    'endpoint_url': MINIO_ENDPOINT
}

print(f"✅ MinIO Config: endpoint={MINIO_ENDPOINT}")

In [ ]:
# Cell 4: Data Paths
# =========================================

DATA_PATH = os.getenv('DATA_PATH', 's3a://winlogbeat/winlogbeat/*.parquet')
BASE_S3_PATH = os.getenv('BASE_S3_PATH', 's3a://temp/c2_detection')
TRAIN_DATA_PATH = f"{BASE_S3_PATH}/train_features.parquet"
MODEL_S3_PATH = f"{BASE_S3_PATH}/model.pth"

print(f"✅ Data Path: {DATA_PATH}")

In [ ]:
# Cell 5: C2 Beacon Detection Parameters
# =========================================

C2_KMEANS_K = int(os.getenv('C2_KMEANS_K', '50'))
C2_KMEANS_SEED = int(os.getenv('C2_KMEANS_SEED', '42'))
C2_AUTOENCODER_INPUT_DIM = int(os.getenv('C2_AUTOENCODER_INPUT_DIM', '10'))
C2_AUTOENCODER_HIDDEN_DIM = int(os.getenv('C2_AUTOENCODER_HIDDEN_DIM', '8'))
C2_AUTOENCODER_LATENT_DIM = int(os.getenv('C2_AUTOENCODER_LATENT_DIM', '3'))
C2_AUTOENCODER_NUM_EPOCHS = int(os.getenv('C2_AUTOENCODER_NUM_EPOCHS', '50'))
C2_BEACON_SCORE_THRESHOLD = float(os.getenv('C2_BEACON_SCORE_THRESHOLD', '0.8'))

print(f"✅ C2 Beacon Config: kmeans_k={C2_KMEANS_K}, autoencoder_epochs={C2_AUTOENCODER_NUM_EPOCHS}, threshold={C2_BEACON_SCORE_THRESHOLD}")

In [ ]:
# Cell 6: Create Spark Session
# =========================================

builder = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)

    .config("spark.driver.host", SPARK_DRIVER_HOST)
    .config("spark.driver.port", str(SPARK_DRIVER_PORT))
    .config("spark.blockManager.port", str(SPARK_BLOCK_MANAGER_PORT))
    .config("spark.ui.port", str(SPARK_UI_PORT))

    .config("spark.executor.cores", str(SPARK_EXECUTOR_CORES))
    .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)

    .config("spark.executor.extraClassPath", SPARK_EXECUTOR_CLASSPATH)
    .config("spark.jars", SPARK_JARS_PATH)
    .config("spark.executor.userClassPathFirst", "true")

    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", MINIO_PATH_STYLE_ACCESS)
)

if USE_GPU:
    builder = (
        builder
        .config("spark.executor.resource.gpu.amount", SPARK_EXECUTOR_GPU_AMOUNT)
        .config("spark.task.resource.gpu.amount", SPARK_TASK_GPU_AMOUNT)
    )

spark = builder.getOrCreate()

print(f"✅ Spark session created: {SPARK_APP_NAME}")
print(f"   Master: {SPARK_MASTER}")
print(f"   GPU: {USE_GPU}")

In [ ]:
# Cell 7: Load Data
# =========================================

df_raw = spark.read.parquet(DATA_PATH)
print(f"✅ Loaded {df_raw.count()} records from MinIO")

## Data Filtering

The following code:
1. Adds a human-readable time column from epoch time
2. Filters data based on needs (Event IDs 1, 3, 22)
3. Splits data into network, DNS, and process contexts

**Key Fields:**
- `event_created`: Event creation time in epoch
- `winlog_event_id`: Sysmon event ID (1, 3, 22)
- `winlog_event_data_processguid`: Unique process session identifier
- `winlog_event_data_destinationip`: Destination IP address (network)
- `winlog_event_data_queryname`: DNS query name (DNS beacons)

In [ ]:
# Cell 8: Filter Data
# =========================================

df_with_time = df_raw.withColumn(
    "readable_time",
    F.from_unixtime(F.col("`event_created`") / 1000000000)
)

CleanData = (
    df_with_time
    .where(F.col("event_code").isin(["1", "3", "22"]))
    .select(
        F.col("event_code"),
        F.col("winlog_user_name"),
        F.col("winlog_computer_name"),
        F.col("event_created"),
        F.col("readable_time"),
        F.col("winlog_event_data_commandLine"),
        F.col("winlog_event_data_currentdirectory"),
        F.col("winlog_event_data_image"),
        F.col("winlog_event_data_parentcommandline"),
        F.col("winlog_event_data_parentimage"),
        F.col("winlog_event_data_parentprocessguid"),
        F.col("winlog_event_data_parentuser"),
        F.col("winlog_event_data_processguid"),
        F.col("winlog_event_data_user"),
        F.col("winlog_event_data_destinationip"),
        F.col("winlog_event_data_destinationport"),
        F.col("winlog_event_data_destinationportname"),
        F.col("winlog_event_data_initiated"),
        F.col("winlog_event_data_protocol"),
        F.col("winlog_event_data_sourcehostname"),
        F.col("winlog_event_data_sourceip"),
        F.col("winlog_event_data_sourceport"),
        F.col("winlog_event_data_queryname"),
        F.col("winlog_event_data_queryresults"),
        F.col("winlog_event_data_querystatus"),
        F.col("winlog_task")
    )
)

network_df = CleanData.filter(
    (F.col("event_code") == 3) &
    (F.col("winlog_event_data_initiated") == True) &
    (F.col("winlog_event_data_destinationip").isNotNull()) &
    (F.col("winlog_event_data_processguid").isNotNull())
)

dns_df = CleanData.filter(
    (F.col("event_code") == 22) &
    (F.col("winlog_event_data_queryname").isNotNull()) &
    (F.col("winlog_event_data_processguid").isNotNull())
)

process_df = CleanData.filter(F.col("event_code") == 1)

print(f"Total Events: {CleanData.count()}")
print(f"Network Events: {network_df.count()}")
print(f"DNS Events: {dns_df.count()}")
print(f"Process Events: {process_df.count()}")

## Sessionization and Feature Engineering

Converts network events into sessions (Process + Destination) and calculates timing statistics.

**Process:**
1. Group events by ProcessGuid
2. Calculate inter-arrival times (deltas between consecutive events)
3. Aggregate timing statistics (mean, stddev, duration)
4. Calculate Coefficient of Variation (CV)

**Result:** Each ProcessGuid becomes a feature vector with timing characteristics

**Note:** CV is key feature - beacons have LOW CV (regular timing), normal web browsing has HIGH CV

In [ ]:
# Cell 9: Sessionize and Extract Timing Features
# =========================================

print("Starting Sessionization and Feature Engineering...")

w = Window.partitionBy("winlog_event_data_processguid").orderBy("event_created")

network_with_lag = network_df.withColumn("prev_time", F.lag("event_created", 1).over(w))

features_df = network_with_lag.groupBy("winlog_event_data_processguid").agg(
    F.count("*").alias("session_count"),
    ((F.max("event_created") - F.min("event_created")) / 1e9).alias("duration_sec"),
    ((F.mean(F.col("event_created") - F.col("prev_time"))) / 1e9).alias("mean_delta_sec"),
    ((F.stddev(F.col("event_created") - F.col("prev_time"))) / 1e9).alias("stddev_delta_sec")
)

features_df = features_df.withColumn(
    "cv",
    F.when(F.col("mean_delta_sec") > 0, F.col("stddev_delta_sec") / F.col("mean_delta_sec")).otherwise(0.0)
)

features_df = features_df.filter(F.col("session_count") > 2)

print(f"✅ Feature engineering complete. {features_df.count()} sessions created")

## Shallow Anomaly Detection (K-Means)

This section uses K-Means clustering to find normal patterns:

**Model:**
- Unsupervised clustering algorithm
- Groups sessions into k clusters (default: 50)
- Each cluster represents a "normal" behavior profile
- Anomaly score = distance to assigned cluster center

**Pipeline:**
1. Data Cleaning: Remove sessions with 0.0 timing (instantaneous bursts/noise)
2. Scaling: K-Means requires data on same scale (StandardScaler)
3. Modeling: Train K-Means on features
4. Scoring: Calculate distance from each point to cluster center

**Interpretation:**
- Low cv + High anomaly_score: Strong candidate for C2 beacon
- High anomaly_score + High cv: Likely unique connection (not beacon)

In [ ]:
# Cell 10: Train K-Means Shallow Model
# =========================================

print("Starting Phase: K-Means Clustering...")

clean_features = features_df.filter(
    (col("mean_delta_sec") > 0) &
    (col("duration_sec") > 0)
)

assembler = VectorAssembler(
    inputCols=["session_count", "duration_sec", "mean_delta_sec", "stddev_delta_sec", "cv"],
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

kmeans = KMeans(k=C2_KMEANS_K, seed=C2_KMEANS_SEED, featuresCol="features", predictionCol="cluster")

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model = pipeline.fit(clean_features)

centers = model.stages[-1].clusterCenters()
sc = spark.sparkContext
broadcast_centers = sc.broadcast(centers)

def calculate_distance(features, cluster_id):
    center = broadcast_centers.value[cluster_id]
    return float(np.linalg.norm(features - center))

distance_udf = udf(calculate_distance, FloatType())

result_df = model.transform(clean_features).withColumn(
    "anomaly_score",
    distance_udf(col("features"), col("cluster"))
)

print(f"✅ K-Means training complete. {result_df.count()} sessions scored")

## Enriching with Process Context

Enriches timing features with process lineage and command-line data for deep model.

**Features:**
- Process image and parent image
- Command line length
- Parent command line length
- Destination port name
- Username

**Approach:**
- Join timing stats with process and network context
- Calculate command line lengths as features
- Handle NULL values appropriately
- Prepare for deep learning model

In [ ]:
# Cell 11: Enrich with Process Context
# =========================================

print("Starting Phase: Context Enrichment...")

process_context = process_df.select(
    "winlog_event_data_processguid",
    "winlog_event_data_commandline",
    "winlog_event_data_parentimage",
    "winlog_event_data_parentcommandline"
).dropDuplicates(["winlog_event_data_processguid"])

network_context = network_df.select(
    "winlog_event_data_processguid",
    "winlog_event_data_destinationip",
    "winlog_event_data_image",
    "winlog_user_name",
    "winlog_event_data_destinationportname"
).dropDuplicates(["winlog_event_data_processguid", "winlog_event_data_destinationip"])

stats_full_context = clean_features.join(
    network_context,
    on=["winlog_event_data_processguid"],
    how="inner"
).join(
    process_context,
    on="winlog_event_data_processguid",
    how="left"
)

stats_full_context = stats_full_context.withColumn("cmdline_len", length(col("winlog_event_data_commandline")))
stats_full_context = stats_full_context.withColumn("parent_cmdline_len", length(col("winlog_event_data_parentcommandline")))

clean_data = stats_full_context.na.fill({
    "cmdline_len": 0,
    "parent_cmdline_len": 0,
    "winlog_event_data_parentimage": "UNKNOWN",
    "winlog_event_data_image": "UNKNOWN",
    "winlog_event_data_destinationportname": "UNKNOWN",
    "winlog_user_name": "UNKNOWN"
})

print(f"✅ Context enrichment complete. {clean_data.count()} sessions enriched")

## Deep Learning Model Training

This section trains a PyTorch Autoencoder for deep anomaly detection:

**Architecture:**
- Input: Rich feature vector (timing + process context)
- Encoder: Compresses to latent space (3 dimensions)
- Decoder: Reconstructs input from latent space
- Training: Minimize reconstruction error (MSE)
- Inference: High reconstruction error = anomalous (candidate beacon)

**Hyperparameters:**
- `INPUT_DIM`: Number of input features (10)
- `HIDDEN_DIM`: Hidden layer dimension (8)
- `LATENT_DIM`: Latent space dimension (3)
- `NUM_EPOCHS`: Training epochs (50)
- Learning rate: 0.001

**Distributed Training:**
- Use TorchDistributor for GPU acceleration
- Workers read training data from MinIO/S3
- Each worker loads PyTorch model on GPU
- Reduces training time significantly

In [ ]:
# Cell 12: Prepare Deep Learning Training Data
# =========================================

print("Starting Phase: Prepare Deep Learning Training Data...")

categorical_cols = [
    "winlog_event_data_image",
    "winlog_user_name",
    "winlog_event_data_destinationportname",
    "winlog_event_data_parentimage"
]

stages = []
for c in categorical_cols:
    indexer = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    stages.append(indexer)

feature_cols = [
    "session_count", "duration_sec", "mean_delta_sec", "stddev_delta_sec", "cv",
    "winlog_event_data_image_idx", "winlog_event_data_parentimage_idx",
    "winlog_event_data_destinationportname_idx",
    "cmdline_len", "parent_cmdline_len"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_deep",
    handleInvalid="keep"
)
stages.append(assembler)

pipeline_full = Pipeline(stages=stages)
model_full = pipeline_full.fit(clean_data)
deep_feature_df = model_full.transform(clean_data)

scaler = StandardScaler(inputCol="features_deep", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(deep_feature_df)
scaled_df = scaler_model.transform(deep_feature_df)

scaled_df.select("features_scaled").write.mode("overwrite").parquet(TRAIN_DATA_PATH)

print(f"✅ Training data saved to {TRAIN_DATA_PATH}")

In [ ]:
# Cell 13: Autoencoder Training Function
# =========================================

def train_autoencoder():
    """
    Train PyTorch Autoencoder on GPU using TorchDistributor.

    Args:
        None (reads from global TRAIN_DATA_PATH)

    Returns:
        None (saves model to MinIO)
    """
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[WORKER] Training on device: {device}")

        df = pd.read_parquet(TRAIN_DATA_PATH, storage_options=STORAGE_OPTIONS)

        X = np.array([v['values'] if isinstance(v, dict) else v for v in df['features_scaled']])
        X_tensor = torch.FloatTensor(X).to(device)

        class Autoencoder(nn.Module):
            def __init__(self, input_dim):
                super(Autoencoder, self).__init__()
                self.encoder = nn.Sequential(
                    nn.Linear(input_dim, C2_AUTOENCODER_HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Linear(C2_AUTOENCODER_HIDDEN_DIM, C2_AUTOENCODER_LATENT_DIM)
                )
                self.decoder = nn.Sequential(
                    nn.Linear(C2_AUTOENCODER_LATENT_DIM, C2_AUTOENCODER_HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Linear(C2_AUTOENCODER_HIDDEN_DIM, input_dim)
                )

            def forward(self, x):
                x = self.encoder(x)
                x = self.decoder(x)
                return x

        model = Autoencoder(X_tensor.shape[1]).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        print(f"[WORKER] Starting Training on GPU... Input shape: {X_tensor.shape}")
        for epoch in range(C2_AUTOENCODER_NUM_EPOCHS):
            optimizer.zero_grad()
            outputs = model(X_tensor)
            loss = criterion(outputs, X_tensor)
            loss.backward()
            optimizer.step()
            if (epoch+1) % 10 == 0:
                print(f"[WORKER] Epoch [{epoch+1}/{C2_AUTOENCODER_NUM_EPOCHS}], Loss: {loss.item():.6f}")

        fs = s3fs.S3FileSystem(**STORAGE_OPTIONS)
        with fs.open(MODEL_S3_PATH, "wb") as f:
            torch.save(model.state_dict(), f)
        print(f"[WORKER] Model saved to {MODEL_S3_PATH}")

    except Exception as e:
        print("[WORKER] ERROR:")
        traceback.print_exc()
        raise e

print("✅ Training function defined")

In [ ]:
# Cell 14: Launch Distributed Training
# =========================================

print("Starting Phase: Distributed Autoencoder Training on GPU...")

distributor = TorchDistributor(
    num_processes=1,
    local_mode=False,
    use_gpu=USE_GPU
)

distributor.run(train_autoencoder)

print("✅ Phase: Distributed Training complete")

## Score Fusion & Final Ranking

Combine shallow (K-Means) and deep (Autoencoder) anomaly scores:

**Scoring Logic:**
1. Load trained Autoencoder model from MinIO
2. Calculate reconstruction error for each session
3. Join shallow and deep scores
4. Normalize both to 0-1 scale (MinMaxScaler)
5. Calculate final beacon score

**Formula:**
beacon_score = norm_shallow * (1 + norm_deep)

- If timing (shallow) is 0, score is 0
- If timing is high and context (deep) is high, score is max
- Accounts for scenarios where deep model detects anomaly but timing is not beacon-like

**Interpretation:**
- High beacon_score: Candidate C2 beacon
- Low beacon_score: Normal traffic

In [ ]:
# Cell 15: Load Model and Calculate Deep Scores
# =========================================

print("Starting Phase: Calculate Deep Anomaly Scores...")

model_s3_path = MODEL_S3_PATH
model_local_path = "model.pth"

conf = spark._jsc.hadoopConfiguration()
uri = spark._jvm.java.net.URI(model_s3_path)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, conf)

src_path = spark._jvm.org.apache.hadoop.fs.Path(model_s3_path)
dst_path = spark._jvm.org.apache.hadoop.fs.Path(model_local_path)

if fs.exists(src_path):
    fs.copyToLocalFile(src_path, dst_path)
else:
    raise FileNotFoundError(f"Model not found at {model_s3_path}")

features_pd = scaled_df.select(
    "features_scaled",
    "winlog_event_data_processguid",
    "winlog_event_data_destinationip",
    "winlog_event_data_image",
    "winlog_event_data_parentimage",
    "winlog_event_data_commandline",
    "winlog_event_data_parentcommandline"
).toPandas()

X = np.array([v['values'] if isinstance(v, dict) else v.toArray() for v in features_pd['features_scaled']])
X_tensor = torch.FloatTensor(X)

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, C2_AUTOENCODER_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(C2_AUTOENCODER_HIDDEN_DIM, C2_AUTOENCODER_LATENT_DIM)
        )
        self.decoder = nn.Sequential(
            nn.Linear(C2_AUTOENCODER_LATENT_DIM, C2_AUTOENCODER_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(C2_AUTOENCODER_HIDDEN_DIM, input_dim)
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

model = Autoencoder(X.shape[1])
model.load_state_dict(torch.load(model_local_path))
model.eval()

with torch.no_grad():
    reconstructed = model(X_tensor)
    errors = torch.mean((X_tensor - reconstructed) ** 2, dim=1).numpy()

deep_score_pd = pd.DataFrame({
    "winlog_event_data_processguid": features_pd['winlog_event_data_processguid'],
    "winlog_event_data_destinationip": features_pd['winlog_event_data_destinationip'],
    "winlog_event_data_image": features_pd['winlog_event_data_image'],
    "winlog_event_data_parentimage": features_pd['winlog_event_data_parentimage'],
    "winlog_event_data_commandline": features_pd['winlog_event_data_commandline'],
    "winlog_event_data_parentcommandline": features_pd['winlog_event_data_parentcommandline'],
    "deep_error": errors
})
deep_score_df = spark.createDataFrame(deep_score_pd)

print(f"✅ Deep scores calculated. {deep_score_df.count()} sessions scored")

In [ ]:
# Cell 16: Fuse Scores and Calculate Final Beacon Score
# =========================================

print("Starting Phase: Score Fusion...")

combined_df = deep_score_df.join(
    result_df.select("winlog_event_data_processguid", "anomaly_score"),
    on=["winlog_event_data_processguid"],
    how="left"
)

assembler_scores = VectorAssembler(inputCols=["anomaly_score", "deep_error"], outputCol="score_vector")
scaler_scores = MinMaxScaler(inputCol="score_vector", outputCol="scaled_score_vector")

pipeline_scores = Pipeline(stages=[assembler_scores, scaler_scores])
score_model = pipeline_scores.fit(combined_df)
scored_df = score_model.transform(combined_df)

extract_shallow = udf(lambda v: float(v[0]), DoubleType())
extract_deep = udf(lambda v: float(v[1]), DoubleType())

final_ranking = scored_df \
    .withColumn("norm_shallow", extract_shallow("scaled_score_vector")) \
    .withColumn("norm_deep", extract_deep("scaled_score_vector")) \
    .withColumn("beacon_score", col("norm_shallow") * (1.0 + col("norm_deep"))) \
    .orderBy(col("beacon_score").desc())

print(f"✅ Score fusion complete. {final_ranking.count()} sessions scored")

## Enrichment with Network Context

Enrich final results with IP addresses and DNS queries for further analysis.

**Enrichment:**
- Aggregate all destination IPs per ProcessGuid
- Aggregate all DNS queries per ProcessGuid
- Join back to final ranking

**Result:** Each beacon candidate shows all IPs/domains it contacted

In [ ]:
# Cell 17: Enrich with Network Context
# =========================================

print("Starting Phase: Network Context Enrichment...")

ip_lookup = network_df.groupBy("winlog_event_data_processguid") \
    .agg(collect_set("winlog_event_data_destinationip").alias("dest_ip_array"))

dns_lookup = dns_df.groupBy("winlog_event_data_processguid") \
    .agg(collect_set("winlog_event_data_queryname").alias("dns_query_array"))

enriched_results = final_ranking \
    .join(ip_lookup, "winlog_event_data_processguid", "left") \
    .join(dns_lookup, "winlog_event_data_processguid", "left")

print(f"✅ Network context enrichment complete. {enriched_results.count()} sessions enriched")

In [ ]:
# Cell 18: Display Top Beacons
# =========================================

print("=== Top 10 C2 Beacon Candidates ===")

enriched_results.select(
    "winlog_event_data_image",
    "winlog_event_data_parentimage",
    "winlog_event_data_commandline",
    "winlog_event_data_parentcommandline",
    "dest_ip_array",
    "dns_query_array",
    "beacon_score",
    "norm_shallow",
    "norm_deep"
).orderBy(F.col("beacon_score").desc()).show(10, truncate=False)

print(f"Total beacons above threshold: {enriched_results.filter(F.col('beacon_score') > C2_BEACON_SCORE_THRESHOLD).count()}")

In [ ]:
# Cell 19: Metrics Helper Function
# =========================================

def calculate_metrics(tp, fp, fn, tn, total_events=None):
    """
    Calculate classification metrics with standard output format.

    Args:
        tp: True Positives
        fp: False Positives
        fn: False Negatives
        tn: True Negatives
        total_events: Optional total for report

    Returns:
        dict with all metrics or None if error
    """
    try:
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        print("\n" + "=" * 40)
        print("CLASSIFICATION METRICS REPORT")
        print("=" * 40)
        if total_events:
            print(f"Total Events Evaluated: {total_events}")
        print(f"True Positives (TP):  {tp:,.0f}")
        print(f"False Positives (FP): {fp:,.0f}")
        print(f"False Negatives (FN): {fn:,.0f}")
        print(f"True Negatives (TN):  {tn:,.0f}")
        print("-" * 40)
        print(f"Accuracy:      {accuracy:.4f}")
        print(f"Precision:     {precision:.4f}")
        print(f"Recall:        {recall:.4f}")
        print(f"F1-Measure:    {f_measure:.4f}")
        print(f"False Pos Rate: {fpr:.4f}")
        print(f"False Neg Rate: {fnr:.4f}")
        print("=" * 40)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f_measure': f_measure,
            'fpr': fpr,
            'fnr': fnr
        }
    except ZeroDivisionError as e:
        print(f"❌ ERROR: Division by zero in metrics calculation: {e}")
        return None

print("✅ Metrics helper function defined")

In [ ]:
# Cell 20: Visualization Helper Function
# =========================================

def plot_confusion_matrix(tp, fp, fn, tn, title='Confusion Matrix'):
    """
    Plot confusion matrix with consistent styling across all notebooks.

    Args:
        tp, fp, fn, tn: Confusion matrix values
        title: Chart title
    """
    matrix_values = [[tp, fn], [fp, tn]]
    color_matrix = [[tp, fn], [fp, tn/200]]

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        color_matrix,
        annot=matrix_values,
        fmt=',.0f',
        cmap='Purples',
        linewidths=2,
        linecolor='white',
        xticklabels=['Predicted Positive', 'Predicted Negative'],
        yticklabels=['Actual Positive', 'Actual Negative'],
        cbar=False,
        annot_kws={"size": 16, "weight": "bold"}
    )

    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('Actual Label', fontsize=14)
    plt.tick_params(labelsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Visualization helper function defined")

In [ ]:
# Cell 21: Calculate Confusion Matrix & Metrics
# =========================================

ScoringMetrics = enriched_results.select(
    "winlog_event_data_image",
    "winlog_event_data_parentimage",
    "winlog_event_data_commandline",
    "winlog_event_data_parentcommandline",
    "dest_ip_array",
    "dns_query_array",
    "beacon_score"
)

target_image = "C:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe"
target_parent = "C:\\Windows\\System32\\svchost.exe"
target_cmd = "\"\"powershell.exe\" -ExecutionPolicy Bypass -File \"C:\\Users\\Public\\health.ps1\"\""

is_malicious_signature = (
    (F.col("winlog_event_data_image").contains("powershell.exe")) &
    (F.col("winlog_event_data_parentimage").contains("svchost.exe")) &
    (F.col("winlog_event_data_commandline").contains("health.ps1"))
)

ScoringMetrics = ScoringMetrics.withColumn(
    "label",
    F.when(is_malicious_signature & (F.col("beacon_score") > C2_BEACON_SCORE_THRESHOLD), "truepositive")
     .when(~is_malicious_signature & (F.col("beacon_score") > C2_BEACON_SCORE_THRESHOLD), "falsepositive")
     .when(is_malicious_signature & (F.col("beacon_score") <= C2_BEACON_SCORE_THRESHOLD), "falsenegative")
     .otherwise(None)
)

total_events = ScoringMetrics.count()
print(f"\nTotal Events Analyzed (N): {total_events}")

tp = 0
fn = 0
tn = 0
fp = 0

labels = ScoringMetrics.select("label").collect()

for row in labels:
    label = row.label
    if label is None:
        tn += 1
    elif label == 'truepositive':
        tp += 1
    elif label == 'falsepositive':
        fp += 1
    elif label == 'falsenegative':
        fn += 1

print(f"\nConfusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

metrics = calculate_metrics(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    total_events=total_events
)

In [ ]:
# Cell 22: Plot Confusion Matrix
# =========================================

plot_confusion_matrix(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    title='Confusion Matrix - C2 Beacon Detection'
)

## Conclusion

This notebook successfully:
- Configured Spark with environment variables (including GPU support)
- Loaded and filtered Sysmon logs (Event IDs 1, 3, 22)
- Sessionized network events and extracted timing features
- Trained K-Means clustering for shallow anomaly detection
- Enriched with process context (command line, parent image)
- Trained PyTorch Autoencoder for deep anomaly detection
- Combined scores using context-weighted fusion formula
- Enriched results with network context (IPs, DNS queries)
- Generated classification metrics and confusion matrix

**Key Insights:**
- Hybrid approach (Clustering + Deep Learning) provides better detection than either alone
- Coefficient of Variation (CV) is key feature for timing regularity
- Context-weighted fusion accounts for behavioral anomalies beyond timing
- Process lineage helps distinguish malicious from benign updaters
- Network context (IPs, DNS queries) enables deeper investigation

**Next Steps:**
- Investigate top beacon candidates by final score
- Review network connections and DNS queries for beacons
- Check process lineage for suspicious parent-child relationships
- Tune hyperparameters (k, autoencoder dimensions, epochs, threshold)
- Update signature matching for your threat landscape
- Consider implementing automated alerts for high-score beacons

**Configuration Changes:**
- Modify `.env` file or set environment variables for:
  - Spark cluster settings (master, GPU flags, memory, cores)
  - MinIO credentials and endpoint
  - Data paths
  - Model hyperparameters:
    - K-Means: k, seed
    - Autoencoder: input_dim, hidden_dim, latent_dim, num_epochs
  - Beacon score threshold (C2_BEACON_SCORE_THRESHOLD)

**Notes:**
- Real-world data is noisy (OneDrive, Chrome, etc.)
- Aggressive tuning of score fusion logic required to reduce false positives
- Signature matching should be updated for your environment
- Threshold of 0.8 used in metrics (adjust via C2_BEACON_SCORE_THRESHOLD)